# tensor-reshape-view — ex2: post-transpose contiguity trap — view raises, reshape works

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-reshape-view`. Running the final beacon cell reports progress against the `PyTorch: reshape vs view` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: reshape vs view` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tensor-reshape-view`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-reshape-view"
DD_SUBTOPIC = "PyTorch: reshape vs view"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `view()` requires contiguous — `reshape()` falls back to a copy

Ex1 picked between `view` and `reshape` based on whether the input was already contiguous. The deepening drill exercises the COMMON failure: after `.transpose()`, the tensor has the right logical shape but the underlying storage is in the wrong order.

```python
x = t.arange(24).reshape(2, 3, 4)
y = x.transpose(0, 2)        # shape (4, 3, 2) but NOT contiguous
y.view(24)                    # RuntimeError: view size is not compatible
y.reshape(24)                 # OK — copies under the hood
y.contiguous().view(24)       # OK — explicit copy then view
```

**Why `view` raises.** `view` requires that the requested shape can be satisfied by re-striding the EXISTING storage. After transpose, elements that are logically adjacent are not physically adjacent — no stride choice works. PyTorch refuses to silently copy and forces you to either call `.contiguous()` first or switch to `.reshape()`.

**`reshape` decides for you.** If a `view` is possible, `reshape` returns one (no copy, same `data_ptr()`). If not, it falls back to `.contiguous().view(...)` (one copy, new `data_ptr()`). This is the safe default — use `view` only when you've already proven contiguity.

### Exercise 2 — post-transpose contiguity trap — view raises, reshape works

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze whether `.view()` raises after a `.transpose()` while `.reshape()` succeeds, and explain the result via `data_ptr()` comparison — `reshape` copies after a non-contiguous transpose.
> Keywords: view, reshape, transpose, contiguous, stride
> ```

**KCs targeted:** `view-raises-on-noncontig`, `reshape-copies-when-needed`

Implement `ex2_post_transpose_audit(x)`. Takes a contiguous tensor `x` of shape `(A, B, C)`. Performs `y = x.transpose(0, 2)` (shape `(C, B, A)`, NOT contiguous in general) and then attempts BOTH `y.view(C*B*A)` and `y.reshape(C*B*A)`.

Return a dict with these keys:

- `'y_contiguous'`: bool — `y.is_contiguous()` immediately after transpose.
- `'view_raised'`: bool — True iff `y.view(C*B*A)` raises `RuntimeError`. False if it succeeded.
- `'reshape_succeeded'`: bool — True iff `y.reshape(C*B*A)` returned a tensor without raising.
- `'reshape_data_ptr_same'`: bool — True iff the reshape's `data_ptr()` equals `y.data_ptr()` (i.e. no copy). False if a copy was made.
- `'reshape_values_correct'`: bool — True iff the reshape's flattened content equals `y.flatten()`.

Constraints:
- Catch `RuntimeError` from the `.view()` attempt — do NOT let it propagate.
- All five keys must be present in the returned dict.

In [ ]:
def ex2_post_transpose_audit(x) -> dict:
    """Audit view-vs-reshape behaviour after transpose."""
    raise NotImplementedError()


def _test_ex2():
    # === Standard non-contiguous case (the headline) ===
    x = t.arange(24).reshape(2, 3, 4).float()
    report = ex2_post_transpose_audit(x)
    assert isinstance(report, dict)
    for k in ('y_contiguous', 'view_raised', 'reshape_succeeded', 'reshape_data_ptr_same', 'reshape_values_correct'):
        assert k in report, f'missing key {k}: {report}'

    assert report['y_contiguous'] is False, f'transposed tensor should NOT be contiguous; got {report}'
    assert report['view_raised'] is True, f'view(numel) on non-contig must raise; got {report}'
    assert report['reshape_succeeded'] is True, f'reshape must succeed; got {report}'
    assert report['reshape_data_ptr_same'] is False, (
        f'after non-contig transpose, reshape must COPY (different data_ptr); got {report}'
    )
    assert report['reshape_values_correct'] is True, f'reshape values mismatch; got {report}'

    # === Already-contiguous corner case: transpose dims of size 1 keeps contig. ===
    # A tensor with shape (1, 3, 1) transposed (0, 2) is still (1, 3, 1) and contiguous.
    x = t.arange(3).reshape(1, 3, 1).float()
    report = ex2_post_transpose_audit(x)
    # In this case y IS contiguous, so view should work and reshape should NOT copy.
    assert report['y_contiguous'] is True, f'(1,3,1) transposed should stay contig; got {report}'
    assert report['view_raised'] is False, f'view on contiguous must succeed; got {report}'
    assert report['reshape_succeeded'] is True
    assert report['reshape_data_ptr_same'] is True, (
        f'reshape on contig should NOT copy (same data_ptr); got {report}'
    )
    assert report['reshape_values_correct'] is True

    # === Larger non-contig case ===
    x = t.arange(60).reshape(3, 4, 5).float()
    report = ex2_post_transpose_audit(x)
    assert report['y_contiguous'] is False
    assert report['view_raised'] is True
    assert report['reshape_succeeded'] is True
    assert report['reshape_values_correct'] is True

    # === Values check: after transpose(0,2), flatten of y should match the
    # reshape that the function produced. ===
    x = t.tensor([[[1., 2.], [3., 4.]], [[5., 6.], [7., 8.]]])  # (2,2,2)
    report = ex2_post_transpose_audit(x)
    assert report['reshape_values_correct'] is True, (
        f'reshape contents must match y.flatten(); got {report}'
    )

    # === Sanity: view_raised + view succeeded are mutually exclusive ===
    for x_test in [t.arange(24).reshape(2,3,4).float(),
                   t.arange(3).reshape(1,3,1).float()]:
        r = ex2_post_transpose_audit(x_test)
        # If view raised, that means view did NOT succeed; vice versa.
        assert isinstance(r['view_raised'], bool)
        assert isinstance(r['reshape_succeeded'], bool)

    # === reshape_succeeded is always True for these inputs (reshape never fails on valid numel) ===
    assert ex2_post_transpose_audit(t.arange(24).reshape(2,3,4).float())['reshape_succeeded'] is True
    assert ex2_post_transpose_audit(t.arange(3).reshape(1,3,1).float())['reshape_succeeded'] is True
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_post_transpose_audit(x):
    A, B, C = x.shape
    y = x.transpose(0, 2)  # (C, B, A) — usually non-contiguous
    n = C * B * A
    report = {'y_contiguous': bool(y.is_contiguous())}

    # Attempt the .view() — catch the RuntimeError it raises on non-contig.
    try:
        _ = y.view(n)
        report['view_raised'] = False
    except RuntimeError:
        report['view_raised'] = True

    # .reshape() never raises for the same numel; may or may not copy.
    try:
        r = y.reshape(n)
        report['reshape_succeeded'] = True
        report['reshape_data_ptr_same'] = (r.data_ptr() == y.data_ptr())
        report['reshape_values_correct'] = bool(t.equal(r, y.flatten()))
    except RuntimeError:
        report['reshape_succeeded'] = False
        report['reshape_data_ptr_same'] = False
        report['reshape_values_correct'] = False
    return report
```

**The asymmetry is by design.** `view` PROMISES no copy — that's its contract. When a no-copy view isn't possible (non-contig storage that the new shape can't be re-strided over), it raises rather than silently copy and break the promise. `reshape` makes no such promise — it falls back to `.contiguous().view(...)` when needed.

**`data_ptr()` is the copy oracle.** Same `data_ptr()` → no copy (view-like). Different `data_ptr()` → copy was made (new storage). After a non-contig transpose, `reshape` always copies; the new data_ptr is your evidence.

**The `(1, 3, 1)` corner case isn't pedantry.** Tensors with singleton dimensions are common (broadcasting, channels-first with batch=1). Their strides interact with transpose in ways that preserve contiguity in some axes — the audit helper exposes this so the user sees that contig is a runtime property, not a shape-derivable one.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()